# Validation of the Interpolation Model

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import random
import shutil

import xarray as xr
from models_plots import (
    plot_daily_error_distribution,
    plot_et_map_comparison,
    plot_et_time_comparison,
    plot_map_error_distribution,
    plot_spatial_distribution_error,
    plot_time_evolution_characteristic_pixels,
    plot_time_evolution_error,
    print_daily_statistics,
    print_pixel_time_stats,
    print_spatial_statistics,
)

from sweat.api import run_window_time_series


In [ ]:
if os.environ.get("SWEAT_TEST_DATA_PATH", None) is None:
    msg = "Variable SWEAT_TEST_DATA_PATH must be set"
    raise ValueError(msg)

In [ ]:
def clean(ts):
    """
    Clean outputs
    """
    # Remove directory
    shutil.rmtree(ts, ignore_errors=True)
    # Recreate directory
    os.makedirs(ts, exist_ok=True)

In [ ]:
def create_et_copy_dir(
    input_dir: str,
    freq: float | None = None,
    cloud_probability: float | None = None,
) -> str:
    """
    Create a directory containing observed evapotranspiration products for interpolation,
    based on a given acquisition frequency and a probability of cloud-contaminated (invalid) dates.

    Parameters
    ----------
    input_dir : str
        Directory where the evapotranspiration and radiation products are stored.
    freq : float, optional
        Acquisition frequency in days.
    cloud_probability : float, optional
        Probability that an observation is invalid.

    Returns
    -------
    str
        Path to the new directory.
    """
    et_original_dir = os.path.join(input_dir, "et")
    et_copy_dir = os.path.join(input_dir, "et_copy")
    shutil.rmtree(et_copy_dir, ignore_errors=True)
    shutil.copytree(et_original_dir, et_copy_dir)
    files = os.listdir(et_copy_dir)
    files.sort()
    if freq is not None:
        files_to_keep = [f for idx, f in enumerate(files) if idx % freq == 0]
        files_to_delete = set(files) - set(files_to_keep)
        for f in files_to_delete:
            os.remove(os.path.join(et_copy_dir, f))
        files = files_to_keep
    if cloud_probability is not None:
        for f in files:
            if random.random() < cloud_probability:  # noqa: S311
                os.remove(os.path.join(et_copy_dir, f))
    return et_copy_dir

We obtained the observed evapotranspiration and daily radiation products using the **prepare_et_single_date** and **prepare_daily_radiation functions** from **ET-DATASET**.

In [ ]:
output_path = "out"
os.makedirs(output_path, exist_ok=True)

In [ ]:
input_dir = os.path.join(
    os.environ.get("SWEAT_TEST_DATA_PATH"), "data_validation", "senegal"
)
time = xr.date_range("2024-03-01", "2024-03-31")
min_date = time[0]
max_date = time[-1]

In [ ]:
dir_sd = os.path.join(input_dir, "et")
dir_rad = os.path.join(input_dir, "daily_radiation")
et_copy_dir = create_et_copy_dir(input_dir, freq=3)

### Output Data

  We then generate the simulated evapotranspiration products using the **run_window_time_series** function from **SWEAT**.

In [ ]:
dir_ts = os.path.join(output_path, "et_timeseries")

In [ ]:
clean(dir_ts)
run_window_time_series(
    period_start=time[0],
    period_end=time[-1],
    et_single_date_dir=et_copy_dir,
    radiation_dir=dir_rad,
    et_time_series_dir=dir_ts,
    window=7,
    shift=1,
)

We run the model for the Coastal Region of Senegal in March 2024, benefiting from ideal meteorological conditions (no rain or clouds). 

In [ ]:
plot_et_map_comparison(dir_sd, dir_ts, "2024-03-05")

In [ ]:
print_daily_statistics("2024-03-05", dir_sd, dir_ts)

### Spatial Distribution of the Error

In [ ]:
plot_spatial_distribution_error(dir_sd, dir_ts, min_date, max_date)

In [ ]:
print_spatial_statistics(dir_sd, dir_ts, min_date, max_date)

In [ ]:
plot_map_error_distribution(dir_sd, dir_ts, min_date, max_date, "mae")

### Time Evolution of the Error

In [ ]:
x = 394479
y = 1539972

In [ ]:
print_pixel_time_stats(dir_sd, dir_ts, min_date, max_date, x, y)

In [ ]:
plot_et_time_comparison(dir_sd, dir_ts, min_date, max_date, x, y)

In [ ]:
plot_time_evolution_error(dir_sd, dir_ts, min_date, max_date, x, y)


In [ ]:
plot_daily_error_distribution(dir_sd, dir_ts, min_date, max_date, x, y, True)

In [ ]:
plot_time_evolution_characteristic_pixels(
    dir_sd, dir_ts, min_date, max_date, absolute=False
)